# Migracao Oracle → ClickHouse

**IMPORTANTE**: Feche o Jupyter completamente e execute no terminal:
```bash
pip install "numpy<2" pandas --force-reinstall
```
Depois abra o Jupyter novamente.

## 1. Verificar NumPy

In [80]:
import numpy as np
print(f"NumPy: {np.__version__}")

NumPy: 1.26.4


## 2. Criar Spark Session (COM Arrow desabilitado)

In [81]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("oracle-clickhouse-migration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark Session criada (Arrow desabilitado)")

Spark Session criada (Arrow desabilitado)


## 3. Configurar ClickHouse JDBC

In [82]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("oracle-clickhouse-migration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark Session criada (Arrow desabilitado)")

Spark Session criada (Arrow desabilitado)


In [83]:
clickhouse_host = "e1a1lieug8.us-central1.gcp.clickhouse.cloud"
clickhouse_port = 8443
clickhouse_user = "default"
clickhouse_password = "_uv765EvWphL_"
clickhouse_database = "raw"

clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_database}?ssl=true"

print(f"ClickHouse JDBC URL: {clickhouse_jdbc_url}")

clickhouse_jdbc_props = {
    "user": clickhouse_user,
    "password": clickhouse_password,
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "ssl": "true",
    "sslmode": "strict"
}

print("ClickHouse configurado para uso com Spark JDBC")

ClickHouse JDBC URL: jdbc:clickhouse://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/raw?ssl=true
ClickHouse configurado para uso com Spark JDBC


## 4. Configurar Oracle


In [99]:
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
jdbc_opts = {
    "url": jdbc_url,
    "user": oracle_user,
    "password": oracle_password,
    "driver": "oracle.jdbc.OracleDriver"
}
print(f"JDBC URL: {jdbc_url}")

JDBC URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br


## 5. Teste de Conexão Oracle


In [100]:
df_test = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("query", "SELECT 1 AS ok FROM dual")
    .load()
)

df_test.show()
print("[OK] Oracle funcionando")

Py4JJavaError: An error occurred while calling o1132.load.
: java.sql.SQLRecoverableException: Listener refused the connection with the following error:
ORA-12514, TNS:listener does not currently know of service requested in connect descriptor
  (CONNECTION_ID=kI93VfXHSHeME5fVQOrv2Q==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at o

## 6. Definir Tabelas para Migração


In [90]:
tables_to_extract = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "ginf.TST_CONTRATOS_BI",
    "ginf.BASE_REGIONAL",
    "ginf.TAB_CIDADE_DELITO_SP_CAP",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.CN9030",
    "siga.SA1030",
    "siga.SA3030",
    "siga.SB1030",
    "siga.SZH030",
    "siga.SZJ030",
    "siga.SZU030",
    "siga.SZV030",
    "siga.SZW030",
    "siga.ZAA030",
    "siga.ZA1030",
    "siga.ZA3030",
    "siga.ZB3030",
    "siga.ZE8030",
    "siga.ZTX030",
    "siga.ZT1030",
    "siga.CN1030",
    "siga.CNB030",
    "siga.SE4030",
    "siga.CNB030",
    "siga.SF2030",
    "siga.ZTX030",
    "ginf.TST_CONTRATOS",
    "scot.ERP_PRODUCT",
    "scot.ERP_PRODUCT_ITEM",
    "scot.ERP_VEHICLE",
    "scot.ERP_AGREEMENT",
    "scot.SC_CITY",
    "scot.SC_GROUP",
    "scot.SC_LOCATION",
    "scot.SC_REQ_FILE",
    "scot.SC_REQUISITION",
    "scot.SC_REQUISITION_HISTORY",
    "scot.SC_REQUISITION_QUEUE",
    "scot.SC_REQUISITION_STATUS",
    "scot.SC_RESERVE",
    "scot.SC_RESERVE_LOCATION",
    "scot.SC_RESULT_CODE",
    "scot.SC_ROLE",
    "scot.SC_STATE",
    "scot.CEPREG",
    "scot.SC_TASK",
    "scot.SC_WAREHOUSE",
    "scot.SC_TECHNICAL_REGISTER",
    "scot.SC_WEBSERVICE_REQUISITION",
    "scot.SC_WEBSERVICE_REQUISITION_HISTORY"
]

# Gerar dicionário ch_tables automaticamente (remove duplicatas)
ch_tables = {}
seen = set()

for table in tables_to_extract:
    schema, name = table.split(".", 1)
    ch_name = name.lower()

    # Evitar duplicatas
    if ch_name not in seen:
        ch_tables[ch_name] = table
        seen.add(ch_name)

print(f"Total de tabelas únicas: {len(ch_tables)}")
print(f"\nPrimeiras 5 tabelas:")
for i, (key, value) in enumerate(list(ch_tables.items())[:5]):
    print(f"  {key} -> {value}")

Total de tabelas únicas: 54

Primeiras 5 tabelas:
  depara_cliente -> ginf.depara_cliente
  base_cep_completa -> ginf.BASE_CEP_COMPLETA
  tst_contratos_bi -> ginf.TST_CONTRATOS_BI
  base_regional -> ginf.BASE_REGIONAL
  tab_cidade_delito_sp_cap -> ginf.TAB_CIDADE_DELITO_SP_CAP


In [91]:
# Verificar quais tabelas já têm dados no ClickHouse (banco raw) e filtrar apenas as vazias

print("=" * 80)
print("VERIFICANDO TABELAS COM E SEM DADOS NO CLICKHOUSE (BANCO RAW)")
print("=" * 80)

tables_with_data = []
tables_without_data = []
tables_not_exist = []

for ch_tbl, oracle_tbl in ch_tables.items():
    try:
        # Tentar contar linhas na tabela ClickHouse no banco RAW
        count_query = f"SELECT COUNT(*) as cnt FROM raw.{ch_tbl}"
        result = client.query(count_query).result_rows

        if result:
            row_count = result[0][0]

            if row_count > 0:
                tables_with_data.append((ch_tbl, oracle_tbl, row_count))
                print(f"✅ raw.{ch_tbl:40s} -> {row_count:>15,} linhas")
            else:
                tables_without_data.append((ch_tbl, oracle_tbl))
                print(f"⚪ raw.{ch_tbl:40s} -> VAZIA (0 linhas)")
    except Exception as e:
        # Tabela não existe
        tables_not_exist.append((ch_tbl, oracle_tbl))
        print(f"❌ raw.{ch_tbl:40s} -> NÃO EXISTE")

print("\n" + "=" * 80)
print("RESUMO")
print("=" * 80)
print(f"Tabelas COM dados:     {len(tables_with_data)}")
print(f"Tabelas VAZIAS:        {len(tables_without_data)}")
print(f"Tabelas NÃO EXISTEM:   {len(tables_not_exist)}")

print("\n" + "=" * 80)
print("TABELAS SEM DADOS (VAZIAS + NÃO EXISTEM)")
print("=" * 80)

# Combinar tabelas vazias e que não existem
tables_to_migrate = tables_without_data + tables_not_exist

if tables_to_migrate:
    for ch_tbl, oracle_tbl in tables_to_migrate:
        print(f"  • {oracle_tbl} -> raw.{ch_tbl}")

    # Criar dicionário apenas com tabelas sem dados
    ch_tables_empty = {ch_tbl: oracle_tbl for ch_tbl, oracle_tbl in tables_to_migrate}

    print(f"\n Total de tabelas para migrar: {len(ch_tables_empty)}")
    print("\nPara migrar apenas estas tabelas, use:")
    print("  ch_tables = ch_tables_empty")
else:
    print("✅ Todas as tabelas já têm dados no ClickHouse (raw)!")

print("=" * 80)


VERIFICANDO TABELAS COM E SEM DADOS NO CLICKHOUSE (BANCO RAW)
✅ raw.depara_cliente                           ->              47 linhas
✅ raw.base_cep_completa                        ->          50,000 linhas
❌ raw.tst_contratos_bi                         -> NÃO EXISTE
✅ raw.base_regional                            ->              27 linhas
✅ raw.tab_cidade_delito_sp_cap                 ->              45 linhas
❌ raw.sc5030                                   -> NÃO EXISTE
❌ raw.sc6030                                   -> NÃO EXISTE
❌ raw.tst_historico_solicitacoes               -> NÃO EXISTE
❌ raw.tst_solicit_cadastradas                  -> NÃO EXISTE
❌ raw.sd2030                                   -> NÃO EXISTE
❌ raw.cn9030                                   -> NÃO EXISTE
❌ raw.sa1030                                   -> NÃO EXISTE
❌ raw.sa3030                                   -> NÃO EXISTE
❌ raw.sb1030                                   -> NÃO EXISTE
❌ raw.szh030                        

## 7. Migração Inicial - RAW/Bronze Layer

In [97]:
import clickhouse_connect
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Manter cliente ClickHouse para gerenciar progresso
client = clickhouse_connect.get_client(
    host=clickhouse_host,
    port=8443,
    username=clickhouse_user,
    password=clickhouse_password,
    database=clickhouse_database,
    secure=True,
    connect_timeout=60,
    send_receive_timeout=300
)
print("ClickHouse client OK (para gerenciar progresso)\n")

# Limite de linhas por batch
MAX_ROWS = 50000

# Criar tabela de progresso no ClickHouse
try:
    client.command("""
        CREATE TABLE IF NOT EXISTS migration_progress
        (
            oracle_table String,
            ch_table String,
            rows_collected Int64,
            total_rows Float64,
            rows_remaining Float64,
            last_id String,
            id_column String,
            status String,
            error String,
            updated_at DateTime DEFAULT now()
        ) ENGINE = ReplacingMergeTree(updated_at)
        ORDER BY oracle_table
    """)
    print("[INFO] Tabela de progresso criada/verificada\n")
except Exception as e:
    print(f"[AVISO] Erro ao criar tabela de progresso: {str(e)[:100]}\n")


def spark_to_ch_type(spark_type):
    """Converte tipo Spark para tipo ClickHouse"""
    tipo = str(spark_type).lower()
    if 'string' in tipo or 'varchar' in tipo:
        base = 'String'
    elif 'decimal' in tipo or 'number' in tipo:
        base = 'String'
    elif 'int' in tipo:
        base = 'Int64'
    elif 'long' in tipo or 'bigint' in tipo:
        base = 'Int64'
    elif 'double' in tipo or 'float' in tipo:
        base = 'Float64'
    elif 'date' in tipo:
        base = 'String'
    elif 'timestamp' in tipo:
        base = 'String'
    elif 'binary' in tipo:
        base = 'String'
    else:
        base = 'String'
    return f'Nullable({base})'


def create_clickhouse_table_from_spark(df, ch_tbl):
    """Cria tabela no ClickHouse baseada no schema do DataFrame Spark"""
    ch_cols = []
    for field in df.schema.fields:
        ch_type = spark_to_ch_type(field.dataType)
        ch_cols.append(f"`{field.name}` {ch_type}")

    create_sql = f"""
        CREATE TABLE IF NOT EXISTS {ch_tbl} (
            {', '.join(ch_cols)}
        ) ENGINE = MergeTree()
        ORDER BY tuple()
    """

    client.command(create_sql)
    return True


def remove_duplicates_spark(df, id_col):
    """
    [RAW/BRONZE LAYER] Mantém dados como estão - SEM REMOÇÃO DE DUPLICATAS

    Na arquitetura Medallion:
    - BRONZE/RAW: Dados brutos do Oracle, exatamente como estão (incluindo duplicatas)
    - SILVER: Dados limpos e transformados (onde a remoção de duplicatas deve ocorrer)
    - GOLD: Dados agregados e prontos para análise

    Esta função retorna o DataFrame original sem modificações.
    """
    print("  [RAW] Dados mantidos como estão (incluindo duplicatas)")
    return df

    # ---- CÓDIGO COMENTADO PARA USO NA SILVER LAYER ----
    # Para remover duplicatas na camada SILVER, descomente o código abaixo:
    #
    if not id_col or id_col not in df.columns:
        # Se não houver coluna ID, usar distinct() em todas as colunas
        original_count = df.count()
        df_clean = df.distinct()
        clean_count = df_clean.count()
        if original_count > clean_count:
            print(f"  [SILVER] {original_count - clean_count} duplicata(s) exata(s) removida(s)")
        return df_clean

    # Usar window function para manter apenas a primeira ocorrência de cada ID
    window_spec = Window.partitionBy(id_col).orderBy(F.monotonically_increasing_id())
    df_dedup = df.withColumn("row_num", F.row_number().over(window_spec)) \
                 .filter(F.col("row_num") == 1) \
                 .drop("row_num")

    original_count = df.count()
    dedup_count = df_dedup.count()

    if original_count > dedup_count:
        print(f"  [SILVER] {original_count - dedup_count} duplicata(s) removida(s)")

    return df_dedup


def save_progress_to_ch(oracle_tbl, ch_tbl, stats):
    """Salva progresso no ClickHouse usando pandas"""
    try:
        progress_data = pd.DataFrame([{
            'oracle_table': oracle_tbl,
            'ch_table': ch_tbl,
            'rows_collected': int(stats.get('rows_collected', 0)),
            'total_rows': float(stats.get('total_rows', 0)),
            'rows_remaining': float(stats.get('rows_remaining', 0)),
            'last_id': str(stats.get('last_id', '')),
            'id_column': str(stats.get('id_column', '')),
            'status': stats.get('status', 'unknown'),
            'error': str(stats.get('error', ''))[:500]
        }])
        client.insert_df('migration_progress', progress_data)
    except Exception as e:
        print(f"  [AVISO] Erro ao salvar progresso: {str(e)[:80]}")


sep = "=" * 60
print(sep)
print(f"MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK")
print(f"Batch size: {MAX_ROWS:,} linhas")
print(sep)

start = datetime.now()
ok = 0
fail = 0
total_rows = 0
failed = []
migration_stats = {}

# Mapear tabelas
ch_tables = {
    "depara_cliente": "ginf.depara_cliente",
    "base_cep_completa": "ginf.BASE_CEP_COMPLETA",
    "bistage": "bistage.TST_CONTRATOS_BI",
    "sc5030": "siga.SC5030",
    "sc6030": "siga.SC6030",
    "tst_historico_solicitacoes": "ginf.TST_HISTORICO_SOLICITACOES",
    "tst_solicit_cadastradas": "ginf.TST_SOLICIT_CADASTRADAS",
    "sd2030": "siga.SD2030",
    "sf2030": "siga.SF2030",
    "ztx030": "siga.ZTX030",
    "tst_contratos": "ginf.TST_CONTRATOS",
    "base_regional":"ginf.base_regional",
    "tab_cidade_delito_sp_cap":"ginf.tab_cidade_delito_sp_cap"
}

for i, (ch_tbl, oracle_tbl) in enumerate(ch_tables.items(), 1):
    pct = (i / len(ch_tables)) * 100
    print(f"\n[{i}/{len(ch_tables)}] ({pct:.1f}%) {oracle_tbl} -> {ch_tbl}")

    try:
        t0 = datetime.now()

        # 1. CONTAR linhas no Oracle
        try:
            df_count = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl}) tmp") \
                .load()
            total_oracle_rows = float(df_count.collect()[0]['CNT'])
            print(f"  Total no Oracle: {total_oracle_rows:,.0f} linhas")
        except Exception as count_err:
            print(f"  [ERRO] Falha ao contar: {str(count_err)[:100]}")
            stats = {
                "status": "failed",
                "error": f"Count failed: {str(count_err)[:100]}",
                "rows_collected": 0,
                "total_rows": 0,
                "rows_remaining": 0,
                "last_id": "",
                "id_column": ""
            }
            migration_stats[oracle_tbl] = stats
            save_progress_to_ch(oracle_tbl, ch_tbl, stats)
            fail += 1
            failed.append(oracle_tbl)
            continue

        # 2. LER dados do Oracle com ORDER BY
        print(f"  Lendo até {MAX_ROWS:,} linhas do Oracle...")
        df = spark.read.format("jdbc") \
            .options(**jdbc_opts) \
            .option("dbtable", f"(SELECT * FROM {oracle_tbl} WHERE ROWNUM <= {MAX_ROWS}) tmp") \
            .load()

        nrows = df.count()

        if nrows == 0:
            print(f"  [AVISO] Tabela vazia")
            stats = {
                "rows_collected": 0,
                "total_rows": total_oracle_rows,
                "rows_remaining": total_oracle_rows,
                "last_id": "",
                "id_column": "",
                "status": "empty",
                "error": ""
            }
            migration_stats[oracle_tbl] = stats
            save_progress_to_ch(oracle_tbl, ch_tbl, stats)
            fail += 1
            failed.append(oracle_tbl)
            continue

        print(f"  Lidas: {nrows:,} linhas")

        # 3. IDENTIFICAR coluna ID
        cols = df.columns
        id_col = None
        for col in cols:
            if 'ID' in col.upper() or col == cols[0]:
                id_col = col
                break

        print(f"  Coluna ID: {id_col}")

        # 4. [RAW LAYER] Manter dados como estão (incluindo duplicatas)
        df_clean = df
        nrows_clean = df_clean.count()

        # 5. CRIAR tabela no ClickHouse
        try:
            # Testar se tabela existe
            client.command(f"SELECT 1 FROM {ch_tbl} LIMIT 1")
            print(f"  Tabela '{ch_tbl}' já existe")
        except:
            print(f"  Criando tabela '{ch_tbl}' no ClickHouse...")
            create_clickhouse_table_from_spark(df_clean, ch_tbl)
            print(f"  [OK] Tabela criada")

        # 6. ESCREVER no ClickHouse usando Spark JDBC
        print(f"  Gravando {nrows_clean:,} linhas no ClickHouse via Spark...")

        df_clean.write \
            .format("jdbc") \
            .option("url", clickhouse_jdbc_url) \
            .option("dbtable", ch_tbl) \
            .option("user", clickhouse_user) \
            .option("password", clickhouse_password) \
            .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
            .option("batchsize", 10000) \
            .option("isolationLevel", "NONE") \
            .mode("append") \
            .save()

        # 7. CAPTURAR último ID
        last_id = None
        if id_col:
            last_row = df_clean.orderBy(F.col(id_col).desc()).first()
            last_id = last_row[id_col] if last_row else None

        dur = (datetime.now() - t0).total_seconds()
        rows_remaining = max(0, total_oracle_rows - nrows_clean)

        print(f"  [OK] {nrows_clean:,} linhas inseridas em {dur:.2f}s")
        print(f"  Restante: {rows_remaining:,.0f} linhas")
        if last_id:
            print(f"  Último ID: {last_id}")

        # 8. SALVAR estatísticas
        stats = {
            "rows_collected": nrows_clean,
            "total_rows": total_oracle_rows,
            "rows_remaining": rows_remaining,
            "last_id": str(last_id) if last_id else "",
            "id_column": id_col or "",
            "status": "complete" if rows_remaining == 0 else "partial",
            "error": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

        ok += 1
        total_rows += nrows_clean

    except Exception as e:
        msg = str(e)[:200]
        print(f"  [ERRO] {msg}")
        fail += 1
        failed.append(oracle_tbl)
        stats = {
            "status": "failed",
            "error": msg,
            "rows_collected": 0,
            "total_rows": 0,
            "rows_remaining": 0,
            "last_id": "",
            "id_column": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

dur = (datetime.now() - start).total_seconds()

print(f"\n{sep}")
print("RESUMO")
print(sep)
print(f"Sucesso: {ok}/{len(ch_tables)}")
print(f"Falhas: {fail}")
print(f"Total linhas coletadas: {total_rows:,}")
print(f"Tempo: {dur:.2f}s")

if failed:
    print(f"\nTabelas com erro:")
    for t in failed:
        print(f"  - {t}")

print(f"\n{sep}")
print("[INFO] Progresso salvo no migration_progress")
print(sep)

ClickHouse client OK (para gerenciar progresso)

[INFO] Tabela de progresso criada/verificada

MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK
Batch size: 50,000 linhas

[1/13] (7.7%) ginf.depara_cliente -> depara_cliente
  [ERRO] Falha ao contar: An error occurred while calling o1045.load.
: java.sql.SQLRecoverableException: Listener refused the

[2/13] (15.4%) ginf.BASE_CEP_COMPLETA -> base_cep_completa
  [ERRO] Falha ao contar: An error occurred while calling o1054.load.
: java.sql.SQLRecoverableException: Listener refused the

[3/13] (23.1%) bistage.TST_CONTRATOS_BI -> bistage
  [ERRO] Falha ao contar: An error occurred while calling o1063.load.
: java.sql.SQLRecoverableException: Listener refused the

[4/13] (30.8%) siga.SC5030 -> sc5030
  [ERRO] Falha ao contar: An error occurred while calling o1072.load.
: java.sql.SQLRecoverableException: Listener refused the

[5/13] (38.5%) siga.SC6030 -> sc6030
  [ERRO] Falha ao contar: An error occurred while calling o1081.load.
: java.sql.SQLRecover

KeyboardInterrupt: 

## 8. Migração Incremental - Continuar de onde parou

In [ ]:
## 8. Migração Incremental - Continuar de onde parou

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime
import time

print("=" * 60)
print("MIGRAÇÃO INCREMENTAL COMPLETA - ORACLE → CLICKHOUSE")
print("=" * 60)
MAX_ROWS = 1000000
# Configurações de performance
MAX_ITERATIONS = 1000  # Máximo de iterações para evitar loop infinito
SLEEP_BETWEEN_BATCHES = 2  # Segundos entre batches (evitar sobrecarga)

iteration = 0
total_migrated = 0

while iteration < MAX_ITERATIONS:
    iteration += 1
    
    print(f"\n{'='*60}")
    print(f"ITERAÇÃO {iteration}/{MAX_ITERATIONS}")
    print(f"{'='*60}")
    
    # Obter progresso do ClickHouse
    progress_query = """
    SELECT 
        oracle_table,
        ch_table,
        rows_collected,
        total_rows,
        rows_remaining,
        last_id,
        id_column,
        status,
        error
    FROM migration_progress
    FINAL
    WHERE status = 'partial'
    ORDER BY rows_remaining DESC  -- Priorizar tabelas maiores
    """
    
    progress_df = client.query_df(progress_query)
    
    if len(progress_df) == 0:
        print("\n" + "="*60)
        print("✅ MIGRAÇÃO COMPLETA - TODAS AS TABELAS FINALIZADAS!")
        print("="*60)
        break
    
    print(f"[INFO] {len(progress_df)} tabela(s) pendente(s)\n")
    
    batch_inserted = 0
    
    for idx, row in progress_df.iterrows():
        oracle_tbl = row['oracle_table']
        ch_tbl = row['ch_table']
        last_id = row['last_id']
        id_col = row['id_column']
        rows_collected = int(row['rows_collected'])
        rows_remaining = float(row['rows_remaining'])
        
        print(f"\n[{idx + 1}/{len(progress_df)}] {oracle_tbl} -> {ch_tbl}")
        print(f"  Progresso: {rows_collected:,} / {rows_collected + rows_remaining:,.0f} linhas")
        print(f"  Restante: {rows_remaining:,.0f} linhas")
        
        try:
            # Validação pré-inserção
            try:
                df_ch_count = spark.read.format("jdbc") \
                    .option("url", clickhouse_jdbc_url) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {ch_tbl}) tmp") \
                    .option("user", clickhouse_user) \
                    .option("password", clickhouse_password) \
                    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                    .load()
                
                ch_count = int(df_ch_count.collect()[0]['cnt'])
                
                if ch_count != rows_collected:
                    print(f"  [AVISO] Ajustando contador: {ch_count:,} linhas no ClickHouse")
                    rows_collected = ch_count
                    
            except Exception as validate_err:
                print(f"  [AVISO] Validação não disponível: {str(validate_err)[:60]}")
            
            t0 = datetime.now()
            
            # Construir query Oracle com paginação correta
            if id_col and last_id:
                query = f"""
                (SELECT * FROM (
                    SELECT * FROM {oracle_tbl} 
                    WHERE {id_col} > '{last_id}'
                    ORDER BY {id_col}
                ) WHERE ROWNUM <= {MAX_ROWS}) tmp
                """
            else:
                query = f"""
                (SELECT * FROM 
                    (SELECT a.*, ROWNUM rnum FROM 
                        (SELECT * FROM {oracle_tbl} ORDER BY 1) a
                     WHERE ROWNUM <= {rows_collected + MAX_ROWS})
                 WHERE rnum > {rows_collected}) tmp
                """
            
            # Ler batch do Oracle
            df = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", query) \
                .option("fetchsize", 10000) \
                .load()
            
            # Remover coluna ROWNUM se existir
            if 'rnum' in df.columns:
                df = df.drop('rnum')
            
            nrows = df.count()
            
            if nrows == 0:
                print(f"  ✅ Migração COMPLETA para esta tabela!")
                stats_complete = pd.DataFrame([{
                    'oracle_table': oracle_tbl,
                    'ch_table': ch_tbl,
                    'rows_collected': rows_collected,
                    'total_rows': float(rows_collected),
                    'rows_remaining': 0.0,
                    'last_id': last_id,
                    'id_column': id_col,
                    'status': 'complete',
                    'error': ''
                }])
                client.insert_df('migration_progress', stats_complete)
                continue
            
            print(f"  Lidas: {nrows:,} linhas")
            
            # [RAW LAYER] Manter dados como estão
            df_clean = df
            
            # Capturar novo último ID
            new_last_id = last_id
            if id_col and id_col in df_clean.columns:
                last_row = df_clean.orderBy(F.col(id_col).desc()).first()
                new_last_id = last_row[id_col] if last_row else last_id
            
            # Escrever no ClickHouse via Spark JDBC
            print(f"  Gravando {nrows:,} linhas no ClickHouse...")
# Antes de gravar, reparticionar para paralelizar a escrita
            num_partitions = 4  # Ajuste conforme necessário

            df_clean.repartition(num_partitions).write \
                .format("jdbc") \
                .option("url", clickhouse_jdbc_url) \
                .option("dbtable", ch_tbl) \
                .option("user", clickhouse_user) \
                .option("password", clickhouse_password) \
                .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                .option("batchsize", 50000) \
                .option("isolationLevel", "NONE") \
                .option("numPartitions", num_partitions) \
                .mode("append") \
                .save()
            
            # Atualizar progresso
            new_total_collected = rows_collected + nrows
            
            # Contar restante
            if id_col and new_last_id:
                df_remaining = spark.read.format("jdbc") \
                    .options(**jdbc_opts) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl} WHERE {id_col} > '{new_last_id}') tmp") \
                    .load()
                rows_remaining = float(df_remaining.collect()[0]['CNT'])
            else:
                rows_remaining = max(0, rows_remaining - nrows)
            
            dur = (datetime.now() - t0).total_seconds()
            print(f"  ✅ {nrows:,} linhas inseridas em {dur:.2f}s")
            print(f"  Total: {new_total_collected:,} | Restante: {rows_remaining:,.0f}")
            
            # Salvar progresso
            new_status = 'complete' if rows_remaining == 0 else 'partial'
            stats_update = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': new_total_collected,
                'total_rows': float(new_total_collected + rows_remaining),
                'rows_remaining': rows_remaining,
                'last_id': str(new_last_id),
                'id_column': id_col,
                'status': new_status,
                'error': ''
            }])
            client.insert_df('migration_progress', stats_update)
            
            batch_inserted += nrows
            total_migrated += nrows
            
        except Exception as e:
            msg = str(e)[:300]
            print(f"  ❌ ERRO: {msg}")
            stats_error = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': rows_collected,
                'total_rows': 0.0,
                'rows_remaining': 0.0,
                'last_id': last_id,
                'id_column': id_col,
                'status': 'error',
                'error': msg[:500]
            }])
            client.insert